# pytest — A Hands-on Walkthrough

This notebook walks through every `pytest` pattern from the lecture, one cell at a time.
**Setup:** if you don't have pytest installed yet, run the next cell.


In [3]:
# Run this once if pytest is not installed

import pytest
import ipytest

# ipytest lets us use pytest inside Jupyter cells.
# If you don't have it: !pip install ipytest

ipytest.autoconfig()

print("pytest version:", pytest.__version__)


pytest version: 9.0.3


---

## The functions we'll be testing

In a real project these would live in their own file (e.g. `mymath.py`) and you'd `import` them. We define them in a cell here for convenience.


In [4]:
def divide(a, b):
    """Return a / b."""
    return a / b


def count_words(text):
    """Count words in a string. Words are separated by whitespace."""
    return len(text.split())


def is_palindrome(s):
    """Return True if s reads the same forwards and backwards (case-insensitive)."""
    s = s.lower()
    return s == s[::-1]


---

## Section 1 — The basics

A pytest test is just a function whose name starts with `test_`. Inside it, you use Python's plain `assert` to check things. If `assert x` is truthy, the test passes; if not, it fails and pytest tells you why.

`%%ipytest` at the top of a cell tells the notebook to run pytest on whatever's inside that cell.


In [5]:
%%ipytest -v

def test_divide_basic():
    assert divide(10, 2) == 5


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py .                                                      [100%]

======================================== 1 passed in 0.01s =========================================


**What just happened:** pytest found one function starting with `test_`, ran it, and reported the result. The `==` comparison evaluated to `True`, so the test passes.

Try a few more:


In [6]:
%%ipytest -v

def test_divide_floats():
    assert divide(5.0, 2.0) == 2.5

def test_divide_negative():
    assert divide(-10, 2) == -5

def test_divide_both_negative():
    assert divide(-10, -2) == 5


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 3 items

t_e7986c6858c94f5a8766397f10ba18da.py ...                                                    [100%]

======================================== 3 passed in 0.02s =========================================


Three tests, three passes. Note pytest collected each one separately — if one fails, the others still run.

### Multiple assertions in one test

You *can* put several asserts in one test, but if the first fails the rest never run. Usually one-test-per-concept is cleaner.


In [7]:
%%ipytest -v

def test_divide_several_cases_in_one():
    assert divide(9, 3) == 3
    assert divide(9, 4) == 2.25
    assert divide(8, 2) == 4


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py .                                                      [100%]

======================================== 1 passed in 0.02s =========================================


---

## Section 2 — Checking for errors

Sometimes the *expected* behavior is for the function to fail. Use `pytest.raises` to assert that a specific error is raised.


In [8]:
%%ipytest -v

def test_divide_by_zero_raises():
    with pytest.raises(ZeroDivisionError):
        divide(5, 0)


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py .                                                      [100%]

======================================== 1 passed in 0.00s =========================================


The block inside `with pytest.raises(...):` is *expected* to fail. If it doesn't fail, the test fails. If it raises a *different* error, the test also fails.

You can also assert on the error message itself:


In [9]:
%%ipytest -v

def test_divide_specific_error_message():
    with pytest.raises(ZeroDivisionError, match="division by zero"):
        divide(1, 0)


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py .                                                      [100%]

======================================== 1 passed in 0.00s =========================================


### A quirky one: booleans

In Python, `True` is `1` and `False` is `0`. So `divide(True, False)` is the same as `divide(1, 0)` — which raises. Worth a separate test because it's surprising.


In [10]:
%%ipytest -v

def test_divide_booleans_quirk():
    # False coerces to 0, so this is a hidden division by zero
    with pytest.raises(ZeroDivisionError):
        divide(True, False)


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py .                                                      [100%]

======================================== 1 passed in 0.00s =========================================


In [11]:
%%ipytest -v

def test_divide_strings_raises_type_error():
    with pytest.raises(TypeError):
        divide("a", "b")


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py .                                                      [100%]

======================================== 1 passed in 0.01s =========================================


---

## Section 3 — Approximate equality (floats)

Floating-point arithmetic is not exact. The classic example: `0.1 + 0.2` does not equal `0.3` in IEEE 754 floats.


In [13]:
# Run this in a regular cell to see the surprise:
print(0.1 + 0.2)

0.30000000000000004


In [14]:
print(0.1 + 0.2 == 0.3)

False


This means a naive `assert` on float equality will fail. Watch:


In [15]:
%%ipytest -v

def test_naive_float_comparison_fails():
    # This test FAILS ON PURPOSE so you can see the failure message
    assert 0.1 + 0.2 == 0.3


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py F                                                      [100%]

============================================= FAILURES =============================================
________________________________ test_naive_float_comparison_fails _________________________________

    def test_naive_float_comparison_fails():
        # This test FAILS ON PURPOSE so you can see the failure message
>       assert 0.1 + 0.2 == 0.3
E       assert (0.1 + 0.2) == 0.3

/var/folders/7y/7h3ljtfd6ml1rb_8ryw5sndm0000gn/T/ipykernel_13671/1518082500.py:3: AssertionError
===================================== short test summary info ======================================
FAILED t_e7986c6858c94f5a8766397f10ba18da.py::test_naive_float

Read the failure message carefully — pytest tells you the assertion line, what `0.1 + 0.2` actually evaluated to (`0.30000000000000004`), and what was expected.

The fix: `pytest.approx()`.


In [16]:
%%ipytest -v

def test_floats_with_approx():
    assert 0.1 + 0.2 == pytest.approx(0.3)

def test_one_third_with_tolerance():
    # You can specify how close 'close enough' is
    assert divide(1, 3) == pytest.approx(0.3333, abs=1e-3)


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 2 items

t_e7986c6858c94f5a8766397f10ba18da.py ..                                                     [100%]

======================================== 2 passed in 0.00s =========================================


---

## Section 4 — `parametrize`: one test, many cases

When you want to run the same test against many inputs, copy-pasting is painful. `parametrize` lets you provide a list of cases, and pytest runs the test once per row, reporting each separately.


In [17]:
%%ipytest -v

@pytest.mark.parametrize("a, b, expected", [
    (10, 2, 5),
    (9, 3, 3),
    (-10, 2, -5),
    (-10, -2, 5),
    (5.0, 2.0, 2.5),
    (0, 5, 0),
])
def test_divide_many_cases(a, b, expected):
    assert divide(a, b) == expected


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 6 items

t_e7986c6858c94f5a8766397f10ba18da.py ......                                                 [100%]

======================================== 6 passed in 0.01s =========================================


Six runs, six pass lines. Each shows the parameter values in brackets — `[10-2-5]`, `[9-3-3]`, etc. — so if one fails you can see exactly which case broke.

### Real partitioning example

Here's `count_words` tested across the partitions we discussed in Exercise 1:


In [23]:
%%ipytest -v

@pytest.mark.parametrize("text, expected", [
    ("hello world", 2),                        # normal case
    ("one", 1),                                # single word
    ("", 0),                                   # empty string
    ("   ", 0),                                # only whitespace
    ("hello\tworld", 2),                      # tab as separator
    ("  leading spaces", 2),                   # leading whitespace
    ("trailing spaces  ", 2),                  # trailing whitespace
    ("multiple   spaces   between", 3),        # multiple spaces
])
def test_count_words_partitions(text, expected):
    assert count_words(text) == expected


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 8 items

t_e7986c6858c94f5a8766397f10ba18da.py ........                                               [100%]

======================================== 8 passed in 0.02s =========================================


This is what a real partitioning test suite looks like — every row exercises a meaningfully different case.


---

## Section 5 — Descriptive test names

The function name **is** the test name. Six months from now, when something breaks, the name is your only clue. Compare:


In [19]:
%%ipytest -v

# GOOD: tells you what's being checked
def test_palindrome_with_mixed_case_returns_true():
    assert is_palindrome("Anna") is True

# GOOD: tells you what's being checked
def test_palindrome_empty_string_is_palindrome():
    assert is_palindrome("") is True

# BAD: tells you nothing if it ever fails
def test_1():
    assert is_palindrome("racecar") is True


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 3 items

t_e7986c6858c94f5a8766397f10ba18da.py ...                                                    [100%]

======================================== 3 passed in 0.01s =========================================


All three pass — but if `test_1` fails next month, you'll have no idea what it was testing without opening the file. The first two tell you immediately.


---

## Section 6 — What a failure looks like

Let's deliberately break a test to see pytest's failure output. **Read this carefully** — when something goes wrong in your real work, this is the message you'll see.


In [20]:
%%ipytest -v

def test_intentional_failure():
    # FAILS ON PURPOSE
    assert divide(10, 2) == 6


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 1 item

t_e7986c6858c94f5a8766397f10ba18da.py F                                                      [100%]

============================================= FAILURES =============================================
_____________________________________ test_intentional_failure _____________________________________

    def test_intentional_failure():
        # FAILS ON PURPOSE
>       assert divide(10, 2) == 6
E       assert 5.0 == 6
E        +  where 5.0 = divide(10, 2)

/var/folders/7y/7h3ljtfd6ml1rb_8ryw5sndm0000gn/T/ipykernel_13671/3468518401.py:3: AssertionError
===================================== short test summary info ======================================
FAILED t_e7986c6858c94f5a8766397f10ba18da.py::test_intentional_failure - assert 5.0 ==

Three things to notice in the failure message:

1. **The test name** at the top of the failure block — `test_intentional_failure`.
2. **The exact assertion line** that failed, with a `>` arrow pointing at it.
3. **The actual value**: pytest annotates `assert 5.0 == 6` with `where 5.0 = divide(10, 2)` so you don't have to guess what the function returned.

This is the entire reason for using a testing framework instead of `if/print`.


---

## Section 7 — Fixtures (a brief intro)

A fixture is a way to provide common setup data to multiple tests. You won't need this for the exercises today, but it's useful to know it exists for when you write larger test suites.


In [21]:
%%ipytest -v

@pytest.fixture
def sample_numbers():
    """This fixture returns a fixed list. Tests can request it by name."""
    return [1, 2, 3, 4, 5]


def test_sum_of_sample(sample_numbers):
    # By naming a parameter `sample_numbers`, this test asks pytest
    # to inject the value the fixture returns.
    assert sum(sample_numbers) == 15


def test_length_of_sample(sample_numbers):
    assert len(sample_numbers) == 5


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 2 items

t_e7986c6858c94f5a8766397f10ba18da.py ..                                                     [100%]

======================================== 2 passed in 0.00s =========================================


Both tests received the same `[1, 2, 3, 4, 5]` from the fixture, without you having to copy-paste it. For real projects this is how you share things like database connections, sample data, or test files across many tests.


---

## Section 8 — Putting it together

Here's a small suite that combines several patterns: parametrized cases, error checking, and clear names. This is what a real test file looks like.


In [22]:
%%ipytest -v

# Happy-path cases
@pytest.mark.parametrize("a, b, expected", [
    (10, 2, 5),
    (9, 4, 2.25),
    (-10, -2, 5),
])
def test_divide_returns_correct_quotient(a, b, expected):
    assert divide(a, b) == expected

# Edge case: zero numerator
def test_divide_zero_numerator_returns_zero():
    assert divide(0, 5) == 0

# Edge case: division by zero
def test_divide_by_zero_raises():
    with pytest.raises(ZeroDivisionError):
        divide(1, 0)

# Floating-point precision case
def test_divide_handles_floats_with_approx():
    assert divide(1, 3) == pytest.approx(0.3333, abs=1e-3)

# Type error case
def test_divide_strings_raises_type_error():
    with pytest.raises(TypeError):
        divide("hello", "world")


======================================= test session starts ========================================
platform darwin -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/zehrausta/PycharmProjects/Testing
plugins: anyio-4.13.0
collected 7 items

t_e7986c6858c94f5a8766397f10ba18da.py .......                                                [100%]

======================================== 7 passed in 0.01s =========================================


Seven tests across five test functions, all in one cell. This is roughly the level of coverage you should aim for on a small function — happy paths, edge cases (zero, division by zero), float handling, and type errors.

---

## Recap — the patterns

| Pattern | Use it for |
|---|---|
| `def test_xxx():` | Mark a function as a test |
| `assert x == y` | Basic equality check |
| `pytest.raises(SomeError)` | The function should fail on this input |
| `pytest.approx(value)` | Floating-point comparison |
| `@pytest.mark.parametrize(...)` | One test, many input cases |
| `@pytest.fixture` | Share setup data across tests |
| Descriptive function names | Future-you's only clue when something fails |

---
